In [1]:
"""
LangChain Agentic Text-to-SQL System with MCP Server Integration
Supports TPCDS schema and natural language query conversion
"""

import os
import json
import csv
from datetime import datetime
from typing import Dict, List, Any, Optional
from pathlib import Path

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, BaseOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.chat_models import ChatOllama
from pydantic import BaseModel, Field
from enum import Enum

import duckdb

In [2]:
with open('./query_example.txt', 'r') as f:
    TPCDS_QUERY_EXAMPLES = f.read()

In [3]:
with open('./schema2.txt', 'r') as f:
    TPCDS_SCHEMA = f.read()

In [4]:
# ============================================================================
# LLM Provider Configuration
# ============================================================================

class LLMProvider(str, Enum):
    """Supported LLM providers"""
    OLLAMA = "ollama"
    OPENAI = "openai"
    CLAUDE = "claude"
    GEMINI = "gemini"


def create_llm(
    provider: LLMProvider,
    model_name: str,
    api_key: Optional[str] = None,
    base_url: Optional[str] = None,
    temperature: float = 0
):
    """
    Create LLM instance based on provider
    
    Args:
        provider: LLM provider (ollama, openai, claude, gemini)
        model_name: Model name/identifier
        api_key: API key (not needed for Ollama)
        base_url: Base URL (for Ollama, default: http://localhost:11434)
        temperature: Temperature for generation
        
    Returns:
        LangChain LLM instance
    """
    if provider == LLMProvider.OLLAMA:
        # Ollama - local model
        ollama_base = base_url or "http://localhost:11434"
        print(f"✓ Connecting to Ollama at {ollama_base}")
        return ChatOllama(
            model=model_name,
            base_url=ollama_base,
            temperature=temperature,
            num_predict=-1
        )
    
    elif provider == LLMProvider.OPENAI:
        # OpenAI API
        if not api_key:
            raise ValueError("OpenAI API key required")
        print(f"✓ Using OpenAI model: {model_name}")
        return ChatOpenAI(
            model=model_name,
            api_key=api_key,
            temperature=temperature
        )
    
    elif provider == LLMProvider.CLAUDE:
        # Anthropic Claude API
        if not api_key:
            raise ValueError("Anthropic API key required")
        print(f"✓ Using Claude model: {model_name}")
        return ChatAnthropic(
            model=model_name,
            api_key=api_key,
            temperature=temperature
        )
    
    elif provider == LLMProvider.GEMINI:
        # Google Gemini API
        if not api_key:
            raise ValueError("Google API key required")
        print(f"✓ Using Gemini model: {model_name}")
        return ChatGoogleGenerativeAI(
            model=model_name,
            google_api_key=api_key,
            temperature=temperature
        )
    
    else:
        raise ValueError(f"Unsupported provider: {provider}")

In [5]:
# ============================================================================
# SQL Query Model
# ============================================================================

class SQLQuery(BaseModel):
    """Structured output for SQL query generation"""
    sql: str = Field(description="The generated SQL query")
    explanation: str = Field(description="Natural language explanation of what the query does")
    tables_used: List[str] = Field(description="List of tables referenced in the query")


# ============================================================================
# Text-to-SQL Prompt Template
# ============================================================================

TEXT_TO_SQL_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are an expert SQL query generator for TPC-DS database schema.

Your task is to convert natural language questions into valid SQL queries.

{schema}

{examples}

## Guidelines:
1. Always use proper JOIN syntax (not implicit joins with WHERE)
2. Use table aliases for better readability
3. Include appropriate WHERE clauses for filtering
4. Use GROUP BY when aggregating data
5. Add ORDER BY for better result presentation
6. Use LIMIT when asking for "top N" results
7. Always reference columns with table aliases (e.g., ss.ss_item_sk)
8. Ensure all foreign key relationships are properly joined
9. Use meaningful column aliases in SELECT
10. Return only valid DuckDB-compatible SQL

## Output Format:
Provide your response as a JSON object with these fields:
- sql: The complete SQL query
- explanation: Brief explanation of what the query does
- tables_used: List of table names used

Generate a query that accurately answers the user's question."""),
    ("human", "{question}")
])


In [6]:
# ============================================================================
# Alternative: Pure Python SQL Executor (Option 2)
# ============================================================================

def execute_sql_pure_python(sql: str, db_path: str = ":memory:") -> Dict[str, Any]:
    """
    Pure Python function to execute SQL without MCP server
    Useful for simpler deployments
    """
    try:
        conn = duckdb.connect(db_path)
        result = conn.execute(sql).fetchall()
        columns = [desc[0] for desc in conn.description]
        conn.close()
        
        return {
            "success": True,
            "rows": result,
            "columns": columns,
            "row_count": len(result)
        }
    except Exception as e:
        return {
            "success": False,
            "error": str(e)
        }


# ============================================================================
# CSV Export Utility
# ============================================================================

class ResultExporter:
    """Export query results to CSV files"""
    
    def __init__(self, output_dir: str = "query_results"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
    
    def export_to_csv(
        self, 
        results: Dict[str, Any], 
        query_text: str,
        filename: Optional[str] = None
    ) -> str:
        """
        Export query results to CSV file
        
        Args:
            results: Query results dictionary
            query_text: Original query text for metadata
            filename: Optional custom filename
            
        Returns:
            Path to saved CSV file
        """
        if not results.get("success"):
            print(f"✗ Cannot export failed query: {results.get('error')}")
            return None
        
        # Generate filename
        if filename is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"query_result_{timestamp}.csv"
        
        filepath = self.output_dir / filename
        
        # Write CSV
        with open(filepath, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            
            # Write metadata as comments
            writer.writerow([f"# Query executed at: {datetime.now().isoformat()}"])
            writer.writerow([f"# Original query: {query_text[:100]}..."])
            writer.writerow([])
            
            # Write headers
            writer.writerow(results['columns'])
            
            # Write data
            writer.writerows(results['rows'])
        
        print(f"✓ Results exported to: {filepath}")
        return str(filepath)


In [7]:
import re

def parse_r1_sql(text: str) -> str:
    """
    Extract the first SQL statement from a DeepSeek-R1 response.
    Strategy:
      - drop all <think>...</think> blocks
      - if a ```sql fenced block``` exists, pull from there
      - else, regex the first WITH/SELECT ... (up to first semicolon or end)
    Returns a SQL string (ends with ';') or raises ValueError.
    """
    # 1) remove any think blocks (non-greedy, DOTALL)
    clean = re.sub(r"(?is)<think>.*?</think>", "", text).strip()

    # 2) prefer fenced code block ```sql ... ```
    m = re.search(r"```(?:sql|SQL)?\s*(.*?)```", clean, flags=re.S)
    if m:
        block = m.group(1).strip()
        sm = re.search(r"(?is)\b(WITH|SELECT)\b.*?(?=;|$)", block)
        if sm:
            return sm.group(0).strip().rstrip(";") + ";"
        # if block already is pure SQL without semicolon, just return it
        if re.search(r"(?is)\b(WITH|SELECT)\b", block):
            return block.rstrip(";") + ";"

    # 3) fallback: inline SQL
    m = re.search(r"(?is)\b(WITH|SELECT)\b.*?(?=;|$)", clean)
    if m:
        return m.group(0).strip().rstrip(";") + ";"

    raise ValueError("No SQL found in the model response.")

class R1SQLParser(BaseOutputParser[str]):
    """LangChain OutputParser wrapper around parse_r1_sql()."""
    def parse(self, text: str) -> str:
        return parse_r1_sql(text)

In [16]:
# ============================================================================
# Main Agent Pipeline
# ============================================================================

class TextToSQLAgent:
    """
    LangChain-based agentic system for text-to-SQL conversion
    Supports multiple LLM providers: Ollama, OpenAI, Claude, Gemini
    """
    
    def __init__(
        self,
        provider: Optional[LLMProvider] = None,
        model_name: Optional[str] = None,
        api_key: Optional[str] = None,
        base_url: Optional[str] = None,
        db_path: str = "tpcds.db",
        temperature: float = 0
    ):
        """
        Initialize the Text-to-SQL agent
        
        Args:
            provider: LLM provider (ollama, openai, claude, gemini). If None, auto-detect from env
            model_name: Model name. If None, use default for provider
            api_key: API key (not needed for Ollama). If None, read from environment
            base_url: Base URL (for Ollama, default: http://localhost:11434)
            use_mcp: Whether to use MCP server or pure Python
            db_path: Database path (default: tpcds.db)
            temperature: Temperature for generation (default: 0)
        """
        # Auto-detect provider if not specified
        self.provider = provider 
        
        # Set default model names for each provider
        self.model_name = model_name 
        
        # Auto-detect API key from environment if not provided
        
        # Create LLM instance based on provider
        self.llm = create_llm(
            provider=self.provider,
            model_name=self.model_name,
            base_url=base_url,
            temperature=temperature
        )
        
        # Store configuration
        self.db_path = db_path
        self.base_url = base_url
        
        # Initialize exporter
        self.exporter = ResultExporter()
        
        # Build the LangChain pipeline
        self.chain = self._build_chain()
        
        print(f"✓ TextToSQLAgent initialized")
        print(f"  Provider: {self.provider}")
        print(f"  Model: {self.model_name}")
        print(f"  Database: {db_path}")
    
    def _build_chain(self):
        """Build the LangChain processing pipeline"""
        
        # Create the chain with structured output
        chain = (
            {
                "question": RunnablePassthrough(),
                "schema": lambda _: TPCDS_SCHEMA,
                "examples": lambda _: TPCDS_QUERY_EXAMPLES
            }
            | TEXT_TO_SQL_PROMPT
            | self.llm
            | StrOutputParser()
        )
            # | R1SQLParser()
        
        return chain
    
    def process_query(
        self, 
        question: str, 
        export_csv: bool = True,
        csv_filename: Optional[str] = None
    ) -> Dict[str, Any]:
        """
        Process natural language query end-to-end
        
        Args:
            question: Natural language question
            export_csv: Whether to export results to CSV
            csv_filename: Optional custom CSV filename
            
        Returns:
            Dictionary with query, results, and metadata
        """
        print(f"\n{'='*60}")
        print(f"Processing query: {question}")
        print(f"{'='*60}\n")
        
        # Step 1: Generate SQL
        print("Step 1: Generating SQL query...")
        sql_query = self.chain.invoke(question)

        # Step 2: Execute SQL
        print("Step 2: Executing SQL query...", sql_query)
        results = execute_sql_pure_python(sql_query, self.db_path)
        
        if results['success']:
            print(f"✓ Query executed successfully ({results['row_count']} rows)")
        else:
            print(f"✗ Query failed: {results['error']}")
            return results
        
        # Step 3: Export to CSV
        csv_path = None
        if export_csv and results['success']:
            print("\nStep 3: Exporting results to CSV...")
            csv_path = self.exporter.export_to_csv(
                results, 
                question,
                csv_filename
            )
        
        # Return comprehensive results
        return {
            "success": True,
            "question": question,
            "sql": sql_query,
            "explanation": explanation,
            "tables_used": tables_used,
            "results": results,
            "csv_path": csv_path
        }

In [17]:
# ============================================================================
# Example Usage
# ============================================================================

def main():
    """Example usage of the TextToSQLAgent"""
    
    agent = TextToSQLAgent(
        provider=LLMProvider.OLLAMA,
        model_name="deepseek-r1:7b",  # or "mistral", "codellama", etc.
        base_url="http://localhost:11434",
        db_path="cube-project/data/tpcds.cb"
    )
    
    # Example queries
    queries = [
        "What are the top 3 selling items by total revenue?",
        "Show me total sales by store for each month in 2024",
        "Which customers have spent more than $1000?",
        "What are the sales for items in the Electronics category?",
    ]
    
    # Process each query
    for i, query in enumerate(queries, 1):
        result = agent.process_query(
            query,
            export_csv=True,
            csv_filename=f"query_{i}_result.csv"
        )
        
        if result['success']:
            print("\n" + "="*60)
            print(f"Results Preview (first 3 rows):")
            print("="*60)
            for row in result['results']['rows'][:3]:
                print(row)
        
        print("\n" + "="*60 + "\n")
        break

In [18]:
main()

✓ Connecting to Ollama at http://localhost:11434
✓ TextToSQLAgent initialized
  Provider: LLMProvider.OLLAMA
  Model: deepseek-r1:7b
  Database: cube-project/data/tpcds.cb

Processing query: What are the top 3 selling items by total revenue?

Step 1: Generating SQL query...
Step 2: Executing SQL query... <think>
Okay, so I need to figure out the top three selling items by total revenue. Hmm, where do I start? Well, first off, I guess I need some data on sales or revenue for different products. But wait, since this is a thought process, maybe I can simulate having that data.

Let me think about what kind of products could be sold and their revenues. Maybe something like electronics, clothing, or home appliances. Each category has various items with different prices and quantities sold. To find the top sellers by total revenue, I need to multiply each item's price by how many were sold (quantity) and then sum that up for all items in a category.

Wait, but without actual numbers, it's ha